In [1]:
from string import punctuation

import pandas as pd

In [10]:
df = pd.read_csv("spam.csv", encoding="latin1")

In [11]:
print(df.shape)

df.head()

(5572, 5)


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 677.1 KB


In [13]:
df = df[['v2', 'v1']].rename(columns={
    'v2' : 'message',
    'v1' : 'result'
})

In [14]:
df.head()

,message,result
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


### Lower-Case Text

In [50]:
df.iloc[5, 0]

'FreeMsg Hey there darling its been 3 weeks now and no word back Id like some fun you up for it still Tb ok XxX std chgs to send å£150 to rcv'

In [51]:
df['message'] = df['message'].str.lower()

In [52]:
df.iloc[5, 0]

'freemsg hey there darling its been 3 weeks now and no word back id like some fun you up for it still tb ok xxx std chgs to send å£150 to rcv'

### Removing HTML tags

In [19]:
df[df['message'].str.contains(r'<.*?>', regex=True, na=False)]

,message,result
689,<Forwarded from 448712404000>Please CALL 08712...,spam
2266,<Forwarded from 88877>FREE entry into our å£25...,spam
2296,<Forwarded from 21870000>Hi - this is your Mai...,spam
2619,<Forwarded from 21870000>Hi - this is your Mai...,spam
4110,URGENT! Your Mobile number has been awarded a ...,spam
5228,PRIVATE! Your 2003 Account Statement for <fone...,spam


In [20]:
import re

def remove_html_tags(text):
    pattern = re.compile(r"<.*?>")
    return pattern.sub(r"", text)

df['message'] = df['message'].apply(remove_html_tags)

In [21]:
df[df['message'].str.contains(r'<.*?>', regex=True, na=False)]

,message,result


### Remove URL

In [25]:
url_df = df[df['message'].str.contains(r'http[s]?://|www\.', regex=True, na=False)]

print(url_df.shape)

print(url_df.iloc[1, 0])

url_df.head()

(106, 2)
XXXMobileMovieClub: To use your credit, click the WAP link in the next txt message or click here>> http://wap. xxxmobilemovieclub.com?n=QJKGIGHJJGCBL


,message,result
12,URGENT! You have won a 1 week FREE membership ...,spam
15,"XXXMobileMovieClub: To use your credit, click ...",spam
163,-PLS STOP bootydelious (32/F) is inviting you ...,spam
190,Are you unique enough? Find out from 30th Augu...,spam
224,"500 New Mobiles from 2004, MUST GO! Txt: NOKIA...",spam


In [26]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r"", text)

df['message'] = df['message'].apply(remove_url)

In [27]:
df[df['message'].str.contains(r'http[s]?://|www\.', regex=True, na=False)]

,message,result


### Treat Emojis

In [28]:
emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)

df[df['message'].apply(lambda x: bool(emoji_pattern.search(str(x))))]

,message,result


### Remove Punctuation

In [31]:
import string

punctuation = string.punctuation

punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [33]:
df['message'].str.contains(f"[{re.escape(punctuation)}]", regex=True, na=False).sum()

np.int64(5103)

In [34]:
# Slower
# def remove_punc(text):
#     for char in punctuation:
#         text = text.replace(char, "")
#     return text

# Faster
def remove_punc(text):
    return text.translate(str.maketrans("", "", punctuation))

In [36]:
df.iloc[1,0]

'Ok lar... Joking wif u oni...'

In [37]:
df['message'] = df['message'].apply(remove_punc)

In [38]:
df.iloc[1,0]

'Ok lar Joking wif u oni'

### Stop Word Removal

In [53]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

In [68]:
sorted(stop_words)[:5]

['a', 'about', 'above', 'after', 'again']

In [63]:
df.iloc[25, 0]

'just forced myself to eat a slice im really not hungry tho this sucks mark is getting worried he knows im sick when i turn down pizza lol'

In [64]:
def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in stop_words])

In [65]:
df['message'] = df['message'].apply(remove_stopwords)

In [66]:
df.iloc[25, 0]

'forced eat slice im really hungry tho sucks mark getting worried knows im sick turn pizza lol'

### Tokenization

In [69]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Jaydeep\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Jaydeep\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [70]:
from nltk.tokenize import word_tokenize

df['tokens'] = df['message'].apply(lambda x: word_tokenize(x.lower()))

In [71]:
df.head()

,message,result,tokens
0,go jurong point crazy available bugis n great ...,ham,"[go, jurong, point, crazy, available, bugis, n..."
1,ok lar joking wif u oni,ham,"[ok, lar, joking, wif, u, oni]"
2,free entry 2 wkly comp win fa cup final tkts 2...,spam,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,u dun say early hor u c already say,ham,"[u, dun, say, early, hor, u, c, already, say]"
4,nah dont think goes usf lives around though,ham,"[nah, dont, think, goes, usf, lives, around, t..."


### Lemmatization

In [72]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Jaydeep\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Jaydeep\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [74]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

df['lemmatized'] = df['tokens'].apply(
    lambda words: [lemmatizer.lemmatize(word, pos='v') for word in words]
)

In [75]:
df.head()

,message,result,tokens,lemmatized
0,go jurong point crazy available bugis n great ...,ham,"[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazy, available, bugis, n..."
1,ok lar joking wif u oni,ham,"[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]"
2,free entry 2 wkly comp win fa cup final tkts 2...,spam,"[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,u dun say early hor u c already say,ham,"[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, early, hor, u, c, already, say]"
4,nah dont think goes usf lives around though,ham,"[nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, go, usf, live, around, though]"


### Join Lemmatize Words back to text message

In [76]:
df['cleaned_message'] = df['lemmatized'].apply(lambda words: " ".join(words))

In [77]:
df.head()

,message,result,tokens,lemmatized,cleaned_message
0,go jurong point crazy available bugis n great ...,ham,"[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazy, available, bugis, n...",go jurong point crazy available bugis n great ...
1,ok lar joking wif u oni,ham,"[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]",ok lar joke wif u oni
2,free entry 2 wkly comp win fa cup final tkts 2...,spam,"[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entry, 2, wkly, comp, win, fa, cup, fin...",free entry 2 wkly comp win fa cup final tkts 2...
3,u dun say early hor u c already say,ham,"[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, early, hor, u, c, already, say]",u dun say early hor u c already say
4,nah dont think goes usf lives around though,ham,"[nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, go, usf, live, around, though]",nah dont think go usf live around though


In [78]:
df = df[['cleaned_message', 'result']]

In [79]:
df.head()

,cleaned_message,result
0,go jurong point crazy available bugis n great ...,ham
1,ok lar joke wif u oni,ham
2,free entry 2 wkly comp win fa cup final tkts 2...,spam
3,u dun say early hor u c already say,ham
4,nah dont think go usf live around though,ham


In [80]:
df.to_csv("Preprocessed_text_data.csv", index=False)